# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# Rebuild the March (feature) / April (label) views from w03_data_contract.

import os
import pathlib
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download


load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

REPO_ID = "FlyRank/internship-warehouse"
FEATURE_MONTH = "2026-03"
LABEL_MONTH = "2026-04"
FEATURE_CUTOFF = pd.Timestamp("2026-03-31")  # nothing dated after this may enter a feature

fact_mar_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN,
    filename=f"fact_content_daily_performance/month={FEATURE_MONTH}/data_0.parquet",
)
fact_apr_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN,
    filename=f"fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet",
)
dim_content_path = hf_hub_download(
    repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN,
    filename="dim_content.parquet",
)

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_mar AS SELECT * FROM read_parquet('{fact_mar_path}')")
con.execute(f"CREATE VIEW fact_apr AS SELECT * FROM read_parquet('{fact_apr_path}')")
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")

print("March rows:", con.sql("SELECT COUNT(*) FROM fact_mar").fetchone()[0])
print("April rows:", con.sql("SELECT COUNT(*) FROM fact_apr").fetchone()[0])

March rows: 9841378
April rows: 10424730


> STEP 1: AGGREGATE March (the feature window) to (client, content) grain
>
> Only March calendar days are summed. gsc_data_available filters out days where GSC simply had no reporting (missing data), which is NOT the same as "zero traffic" — collapsing the two would quietly bias every impression/click sum down.

In [3]:
mar_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS march_impressions,
        SUM(gsc_clicks)                                                 AS march_clicks,
        AVG(gsc_avg_position)                                           AS march_avg_position,
        SUM(ga4_sessions)                                               AS march_sessions,
        SUM(ga4_engaged_sessions)                                       AS march_engaged_sessions,
        SUM(ga4_total_engagement_sec)                                   AS march_engagement_sec,
        SUM(scroll_events)                                              AS march_scroll_events,
        SUM(sessions_ai)                                                AS march_ai_sessions,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_data_available)   AS march_gsc_days,
        COUNT(DISTINCT report_date)                                     AS march_days_present
    FROM fact_mar
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Unique (client, content) pairs in March:", len(mar_agg))
mar_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique (client, content) pairs in March: 331437


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_sessions,march_engaged_sessions,march_engagement_sec,march_scroll_events,march_ai_sessions,march_gsc_days,march_days_present
0,client_625b6439094e23e4,content_cb688c7000658857,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0,31
1,client_625b6439094e23e4,content_8f04218ef9d8cdf9,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0,31
2,client_625b6439094e23e4,content_8c051a8d1050ec4f,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0,31
3,client_625b6439094e23e4,content_2f2e8cae9100a76b,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0,31
4,client_625b6439094e23e4,content_cb7d03d2163d4ecd,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0,31


> Step 2: Aggregate April (the label window ) - clicks only


In [4]:
apr_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM fact_apr
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Unique (client, content) pairs in April:", len(apr_agg))

Unique (client, content) pairs in April: 362172


> Step 3: dim_content is a CURRENT-STATE snapshot, not a point-in-time table ─ content_updated_date / last_optimized_date / optimization_eligible_date can describe something that happened in April or later (the snapshot was pulled "now", not "as of March 31"). Using them unmodified would be a textbook future/overlapping-window leak. We prove this and mask it in one place.

In [5]:
dim_raw = con.sql("SELECT * FROM dim_content").df()

date_cols = ["content_updated_date", "last_optimized_date", "optimization_eligible_date"]
for date_col in date_cols:
    dim_raw[date_col] = pd.to_datetime(dim_raw[date_col], errors="coerce")
    # Anything dated after the cutoff did not exist "as of March 31" -> mask it out
    dim_raw.loc[dim_raw[date_col] > FEATURE_CUTOFF, date_col] = pd.NaT

dim_raw["content_created_date"] = pd.to_datetime(dim_raw["content_created_date"], errors="coerce")
dim_raw["content_age_days"] = (FEATURE_CUTOFF - dim_raw["content_created_date"]).dt.days
dim_raw["days_since_last_optimized"] = (FEATURE_CUTOFF - dim_raw["last_optimized_date"]).dt.days
# Never optimized before March 31 (still NaT after masking) -> "days since last
# optimize" is effectively the content's whole age, not an unknown/NaN.
dim_raw["days_since_last_optimized"] = dim_raw["days_since_last_optimized"].fillna(dim_raw["content_age_days"])

dim_features = dim_raw[[
    "client_hash_id", "content_hash_id",
    "word_count", "char_count", "keyword_char_count", "keyword_token_count",
    "search_volume", "competition", "competition_level", "cpc", "main_intent",
    "content_type", "backlinks", "category_count", "provider_used", "model_used",
    "content_age_days", "days_since_last_optimized",
]].copy()

dim_features.head()

,client_hash_id,content_hash_id,word_count,char_count,keyword_char_count,keyword_token_count,search_volume,competition,competition_level,cpc,main_intent,content_type,backlinks,category_count,provider_used,model_used,content_age_days,days_since_last_optimized
0,client_04660893ae39614a,content_004de9653278b5a4,2555,15682,22,4,30,0.91,HIGH,0.98,transactional,keyword article,16,3,gemini-generate-content,gemini-3-flash-preview,-60,-60.0
1,client_04660893ae39614a,content_00dc5efae381b2ab,2430,15438,31,6,10,0.00,LOW,0.00,commercial,keyword article,0,4,gemini-generate-content,gemini-3-flash-preview,-73,-73.0
2,client_04660893ae39614a,content_01410f2556c327ac,2645,16576,22,5,480,0.36,MEDIUM,0.62,informational,keyword article,169,4,gemini-generate-content,gemini-3-flash-preview,-39,-39.0
3,client_04660893ae39614a,content_019f27f634053ca7,2522,15457,14,3,0,0.00,LOW,0.00,transactional,keyword article,0,4,gemini-generate-content,gemini-3-flash-preview,-76,-76.0
4,client_04660893ae39614a,content_01efa71faea45dcc,2552,15776,24,6,2400,0.70,HIGH,0.90,transactional,keyword article,52,4,gemini-generate-content,gemini-3-flash-preview,-51,-51.0


> Step 4: Assemble the raw feature frame (March signals + static content traits) 

In [6]:
feat_df = mar_agg.merge(dim_features, on=["client_hash_id", "content_hash_id"], how="left")

# Ratios derived FROM March-only sums are still legal March-only features —
# they don't introduce any new information, just reshape what's already there.
feat_df["march_ctr"] = np.where(
    feat_df["march_impressions"] > 0, feat_df["march_clicks"] / feat_df["march_impressions"], 0.0
)
feat_df["march_engagement_rate"] = np.where(
    feat_df["march_sessions"] > 0, feat_df["march_engaged_sessions"] / feat_df["march_sessions"], 0.0
)
feat_df["march_ai_traffic_pct"] = np.where(
    feat_df["march_sessions"] > 0, feat_df["march_ai_sessions"] / feat_df["march_sessions"] * 100, 0.0
)

print("Raw feature frame shape:", feat_df.shape)
feat_df.head()

Raw feature frame shape: (331437, 31)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_sessions,march_engaged_sessions,march_engagement_sec,march_scroll_events,march_ai_sessions,...,content_type,backlinks,category_count,provider_used,model_used,content_age_days,days_since_last_optimized,march_ctr,march_engagement_rate,march_ai_traffic_pct
0,client_625b6439094e23e4,content_cb688c7000658857,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,keyword article,<NA>,0,None,None,332,332.0,0.0,0.0,0.0
1,client_625b6439094e23e4,content_8f04218ef9d8cdf9,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,keyword article,<NA>,0,None,None,332,332.0,0.0,0.0,0.0
2,client_625b6439094e23e4,content_8c051a8d1050ec4f,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,keyword article,<NA>,0,None,None,332,332.0,0.0,0.0,0.0
3,client_625b6439094e23e4,content_2f2e8cae9100a76b,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,keyword article,<NA>,0,None,None,332,332.0,0.0,0.0,0.0
4,client_625b6439094e23e4,content_cb7d03d2163d4ecd,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,keyword article,<NA>,0,None,None,332,332.0,0.0,0.0,0.0


> every column built above is ,either a march-only-sum, a ratio of two march only sums or a static content trait masked to macrch 31. nothing from april has entered this frame yet

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.